To do:

- Hyperparameter tuning (Alpha, learning rate, batch size so on - not sure how to figure this out)
    - There is precedence for no hyperparameter tuning from the author of the OG NLI model that DEBATE is based on = Due to computational restrains and the points from this paper, no hyperparameter tuning was performed in this case. The model tuning in itself is also not the primary focus in this paper, but simply serves as a tool for the actual inquiry into blame in the Danish Parliament


In [ ]:
#%pip install -r "requirements_bert.txt"

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

In [2]:
import gc
import os
import torch
import pandas as pd
from types import SimpleNamespace
from datasets import Dataset
# This is truly a script of all time - import takes a long time because of my init function :)
from scripting_training import ModelInstantiation, load_data, WeightedLossTrainer, read_jsonl
from sklearn.metrics import classification_report

/work/RuneEgeskovTrust#9638/miniconda3/envs/blame/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-11 23:01:17.408999: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
# This replaces argparse variables
args = SimpleNamespace(
    output_path=Path("../../outputs"),
    validation_input_path=Path("/work/RuneEgeskovTrust#9638/data/3_5_agreement.jsonl"),
    save_model=False
)

datasets = [ # Obviously five different datasets in future, doing this to test the loop
    {"path": Path("/work/RuneEgeskovTrust#9638/data/3_5_agreement.jsonl"), "model_name": "data_1"},
    {"path": Path("/work/RuneEgeskovTrust#9638/data/3_5_agreement.jsonl"), "model_name": "data_1_2"},
    {"path": Path("data/dataset_1_2_3_template.jsonl"), "model_name": "data_1_2_3"},
    {"path": Path("data/dataset_1_2_3_4_template.jsonl"), "model_name": "data_1_2_3_4"},
    {"path": Path("data/dataset_1_2_3_4_5_template.jsonl"), "model_name": "data_1_2_3_4_5"},
]

In [4]:
for ds in datasets:
    print(f"\n\n#### Training on {ds['model_name']} ####\n\n")
    
    model = ModelInstantiation()
    
    output_model_dir = os.path.join("..", args.output_path) # This ends up in member files outside git
    os.makedirs(output_model_dir, exist_ok=True)

    tokenized_eval, tokenized_train, class_weights = load_data(
        model, input_path=ds["path"]
    )

    trainer = WeightedLossTrainer(
        model=model.lora_model,
        args=model.training_setup(
            output_path_checkpoints=f'{output_model_dir}/checkpoints/{ds["model_name"]}'
        ),
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        compute_metrics=model.compute_metrics,
        class_weights=class_weights
    )
    # Training
    trainer.train()

    # This is very inefficient, but it is late
    val_records = read_jsonl(args.validation_input_path)
    val = pd.DataFrame(val_records)
    val = val[["text","label"]] # Necessary to convert to dataframe for the following operations
    val.rename(columns={'label': 'labels'}, inplace=True)
    val_dataset = Dataset.from_pandas(val)
    tokenized_val = val_dataset.map(model.tokenize_function, batched=True)

    # Running predictions, should use best checkpoint (weighted BCE) per default
    predictions_output = trainer.predict(tokenized_val)
    logits = predictions_output.predictions
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    preds = probs[:, 1].round().astype(int)
    labels = predictions_output.label_ids

    print(classification_report(labels, preds, target_names=["Label 0", "Label 1"]))

    if args.save_model:
        trainer.model.save_pretrained(
            f'{output_model_dir}/full_models/{ds["model_name"]}'
        )
    else:
        print("Model not saved")

    # Clearing model to avoid OOM issues
    del model, trainer, tokenized_train, tokenized_eval
    torch.cuda.empty_cache()
    gc.collect()



#### Training on data_1 ####






#### Running training on cpu ####




Loading weights: 100%|██████████| 136/136 [00:00<00:00, 38778.07it/s]
ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 13,664,384 || all params: 321,196,162 || trainable%: 4.2542


#### Reading "/work/RuneEgeskovTrust#9638/data/3_5_agreement.jsonl" for training / test data (90/10 split) ####




Map: 100%|██████████| 39/39 [00:00<00:00, 2420.26 examples/s]
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/work/RuneEgeskovTrust#9638/miniconda3/envs/blame/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss




#### Reading "/work/RuneEgeskovTrust#9638/data/3_5_agreement.jsonl" for training / test data (90/10 split) ####




Map: 100%|██████████| 44/44 [00:00<00:00, 2451.51 examples/s]
/work/RuneEgeskovTrust#9638/miniconda3/envs/blame/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


NameError: name 'train_dataset' is not defined

In [ ]:
base_model_name = "jhu-clsp/mmBERT-base"

base_model = AutoModelForSequenceClassification.from_pretrained(
    base_model_name,
    num_labels=2,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

In [ ]:
lora_config = LoraConfig(
    r=32,  # Low-rank dimension
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules="all-linear"  # Fine-tuning all linear (classification, attention... layers)
)

lora_model = prepare_model_for_kbit_training(base_model)

# Enable gradient computation for LoRA parameters
for name, param in lora_model.named_parameters():
    if 'lora' in name.lower():
        param.requires_grad = True

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], 
    padding="max_length", 
    truncation=True,
    max_length=512, # Padding to 512 to massively cut down on computation compared to base 8,192 tokens. 
    )

In [ ]:
val_dataframe = pd.read_json("/work/MarkusLundsfrydJensen#1865/Bachelor_project/Model_data/validation_set.json")

val_dataframe = val_dataframe[['text', 'label']]

val_dataframe.rename(columns={'label': 'labels'}, inplace=True)

val_dataset = Dataset.from_pandas(val_dataframe)
tokenized_val = val_dataset.map(tokenize_function, batched=True, num_proc=16)


In [ ]:
input_data_dir = output_path_json

dataframe_3_4_5 = pd.read_json(input_data_dir)

dataframe_3_4_5 = dataframe_3_4_5.sample(frac=1)

test_dataframe = dataframe_3_4_5[['text', 'label']]

test_dataframe.rename(columns={'label': 'labels'}, inplace=True)

test_dataset = Dataset.from_pandas(test_dataframe)

tokenized_test = test_dataset.map(tokenize_function, batched=True, num_proc=16)

In [ ]:
# 1. Smaller batch size with gradient accumulation
training_args = TrainingArguments(
    output_dir='/work/MarkusLundsfrydJensen#1865/Checkpoints_markus',
    learning_rate=1e-5, #Do minimalistic grid search
    num_train_epochs=3,
    per_device_train_batch_size=16,  # Reduced
    gradient_accumulation_steps=16,  # Effective batch = 256
    logging_steps=1,  # More frequent logging
    eval_strategy="steps",  # Evaluate more frequently
    save_strategy="steps",
    eval_steps = 500,
    save_steps = 500,
    dataloader_pin_memory=True,
    dataloader_num_workers=8,
    remove_unused_columns=True,
    max_grad_norm=1.0,
    disable_tqdm=False,
    load_best_model_at_end=True,  # Add this
    metric_for_best_model="f1", # Add this
    greater_is_better = True
      
)

#OBS implement early stopping and more epochs, lower learning rate?

In [ ]:
# Calculate class weights
label_counts = test_dataframe['labels'].value_counts()
total = len(test_dataframe)
weight_for_0 = total / (2 * label_counts[0])
weight_for_1 = total / (2 * label_counts[1])

print(f"Class 0 weight: {weight_for_0}")
print(f"Class 1 weight: {weight_for_1}")
print(f"Label distribution: {label_counts}")

In [ ]:
def weighted_bincrossentropy(true, pred, weight_zero = weight_for_0, weight_one = weight_for_1):
    """
    Calculates weighted binary cross entropy. The weights are fixed to represent class imbalance in the dataset.
        
    For example if there are 10x as many positive classes as negative classes,
        if you adjust weight_zero = 1.0, weight_one = 0.1, then false positives 
        will be penalized 10 times as much as false negatives.

    """
  
    # calculate the binary cross entropy
    bin_crossentropy = binary_crossentropy(true, pred)
    
    # apply the weights
    weights = true * weight_one + (1. - true) * weight_zero
    #weights /= (weight_one + weight_zero) # Normalizing to be more consistent with regular BCE for comparison 
    weighted_bin_crossentropy = weights * bin_crossentropy 

    return np.mean(weighted_bin_crossentropy)

In [ ]:

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    #From logits to probabilities
    probs_2d = np.exp(predictions) / np.exp(predictions).sum(axis=1, keepdims=True)
    probs = probs_2d[:, 1]  # positive class extraction
    
    weigthted_bce = weighted_bincrossentropy(labels, probs)
    keras_bce = binary_crossentropy(labels, probs)
    keras_bce = float(np.mean(keras_bce.numpy()))  # Converting from keras eagertensor to float value
    
    # Wrapping all metrics to floats for json serialization during model eval
    return {
        'keras_BCE': keras_bce,
        'weighted BCE': weigthted_bce,
        'recall': float(recall_score(labels, probs.round())),
        'precision': float(average_precision_score(labels, probs.round())), #OBS CHANGE THIS
        'accuracy': float(accuracy_score(labels, probs.round())), # Need rounding for these two computations (integer required)
        'f1': float(f1_score(labels, probs.round(), average='macro')), # macro f1 is better for imbalanced dataset
        'number_of_true_preds': sum(probs.round()),
        'number_of_true_labels': sum(labels)
    }

In [ ]:
# Create custom trainer with weighted loss

from torch import nn
import torch

# Calculate class weights
label_counts = test_dataframe['labels'].value_counts()
total = len(test_dataframe)
weight_for_0 = total / (2 * label_counts[0])
weight_for_1 = total / (2 * label_counts[1])

print(f"Class 0 weight: {weight_for_0}")
print(f"Class 1 weight: {weight_for_1}")
print(f"Label distribution: {label_counts}")

class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Create the class weights tensor

#full tuning
class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float32)
print(class_weights)
# Use the custom trainer instead of the regular Trainer
trainer = WeightedLossTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_test,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    class_weights=class_weights  # Pass the weights here
)

trainer.train()

In [ ]:
# Merge (now clean, no quantization artifacts)
model = trainer.model
merged_model = model.merge_and_unload()

output_model_dir = "/work/MarkusLundsfrydJensen#1865/Full_fine_tune_markus/full_trained_model_v3"
merged_model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

In [ ]:
#Hold up

In [ ]:
#OBS need to be made usable
class BlameDetectorDa_v0(object):

    def __init__(self, model, tokenizer, max_length):

        self.model = model
        self.max_length = max_length
        self.tokenizer = tokenizer
        #self.batch_size = batch_size

        self.model_initialization()

        return

    def model_initialization(self):
        #self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_path)
        self.model.eval()
            
        # Move to GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(self.device)
        self.model.to(self.device)

        print(f"Model loaded successfully on {self.device}")

        return

    def predict(self):
        """Make a prediction on a single text input."""
        # Tokenize input
        inputs = self.tokenizer(
            self.text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Move inputs to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Make prediction
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probabilities = torch.softmax(logits, dim=1)
            predicted_class = torch.argmax(probabilities, dim=1).item()
            confidence = probabilities[0][predicted_class].item()
        
        return predicted_class, confidence, probabilities[0].cpu().numpy()

    def run_prediction(self, text):

        self.text = text
        predicted_class, confidence, probs = self.predict()
            
        return predicted_class, confidence


In [ ]:
BD = BlameDetectorDa_v0(model = model, tokenizer = tokenizer, max_length = 512)

In [ ]:
import json
json_path = "/work/MarkusLundsfrydJensen#1865/Bachelor_project/Model_data/validation_set.json"

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
for entry in data:
    text = entry["text"]

    entry["predicted_class"], entry["confidence"] = BD.run_prediction(text)

In [ ]:
true_labels = 0
pred_true = 0
false_labels = 0
pred_false = 0

correct_pred = 0
incorrect_pred = 0

for entry in data:
    if entry["label"] == 1:
        true_labels += 1
    if entry["predicted_class"] == 1:
        pred_true +=1

    if entry["label"] == 0:
        false_labels += 1
    if entry["predicted_class"] == 0:
        pred_false +=1

    if entry["label"] == entry["predicted_class"]:
        correct_pred +=1

    if entry["label"] != entry["predicted_class"]:
        incorrect_pred +=1


print(correct_pred/len(data))

In [ ]:
true_labels = [entry["label"] for entry in data]

predictions= [entry["predicted_class"] for entry in data]

evaluation = (predictions, true_labels)

from sklearn.metrics import accuracy_score, f1_score, average_precision_score, recall_score, classification_report, confusion_matrix

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # Wrapping all metrics to floats for json serialization during model eval
    return {
        'recall': float(recall_score(labels, predictions)),
        'precision': float(average_precision_score(labels, predictions)),
        'accuracy': float(accuracy_score(labels, predictions)), # Need rounding for these two computations (integer required)
        'f1': float(f1_score(labels, predictions, average='macro')), # macro f1 is better for imbalanced dataset
        'number_of_true_preds': sum(predictions),
        'number_of_true_labels': sum(labels)
    }

compute_metrics(evaluation)